# 00 — Test complet du GPU T4 (Colab) et exposition de l'API

**Objectif (S1, R2 + R5)** : prouver par des mesures que le plan est réalisable sur un T4 gratuit, avant de construire dessus.

| Test | Critère go / no-go |
| --- | --- |
| GPU attribué, capacité 7.5, pas de BF16 natif | T4 confirmé → FP16 (décision D-004) |
| Qwen3-4B en FP16 | VRAM < 15 Go, vitesse mesurée |
| Qwen3-8B en 4 bits | VRAM < 15 Go, vitesse mesurée |
| QLoRA 50 pas + checkpoint sur Drive | Pas de dépassement mémoire |
| Coupure de session puis reprise | L'entraînement repart du pas 50 |
| API FastAPI + tunnel | `GET /health` répond depuis l'extérieur en < 10 s |

Les résultats sont écrits dans `t4_report.json` sur Drive : à recopier dans `docs/decisions.md`.

**Avant de commencer** : *Exécution → Modifier le type d'exécution → GPU T4*.

In [ ]:
# 1. GPU et capacités
!nvidia-smi
import json, time, torch
report = {"date": time.strftime("%Y-%m-%d %H:%M")}
assert torch.cuda.is_available(), "Pas de GPU : changez le type d'exécution"
name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
report["gpu"] = {"nom": name, "capacite": f"{cap[0]}.{cap[1]}", "vram_go": round(total_gb, 1),
                 "torch": torch.__version__}
print(report["gpu"])
if cap < (8, 0):
    print("→ Capacité < 8.0 : FP16 obligatoire, ni BF16 ni FlashAttention-2 (D-004).")

In [ ]:
# 2. Drive pour les checkpoints et le rapport (survivent à une coupure)
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/ifrs17_copilot'
CKPT = f'{BASE}/checkpoints/qlora_test'
os.makedirs(CKPT, exist_ok=True)

# Après une coupure : on reprend le rapport déjà enregistré au lieu de l'écraser
if os.path.exists(f'{BASE}/t4_report.json'):
    report = {**json.load(open(f'{BASE}/t4_report.json')), **report}

def save_report():
    with open(f'{BASE}/t4_report.json', 'w') as f:
        json.dump(report, f, indent=2, ensure_ascii=False)
    print('rapport enregistré :', f'{BASE}/t4_report.json')

In [ ]:
# 3. Bibliothèques (torch est déjà installé sur Colab)
!pip install -q -U transformers accelerate bitsandbytes peft datasets

In [ ]:
# 4. Outil de mesure : VRAM maximale et tokens/s
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def free():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

PROMPT = "Explique en trois phrases ce qu'est la marge sur services contractuels (CSM) dans IFRS 17."

@torch.inference_mode()
def bench(model, tok, new_tokens=128):
    msgs = [{"role": "user", "content": PROMPT}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                   enable_thinking=False)   # Qwen3 : pas de mode « réflexion »
    ids = tok(text, return_tensors="pt").to(model.device)
    model.generate(**ids, max_new_tokens=8)                  # échauffement
    torch.cuda.synchronize(); t0 = time.perf_counter()
    out = model.generate(**ids, max_new_tokens=new_tokens, do_sample=False)
    torch.cuda.synchronize(); dt = time.perf_counter() - t0
    n = out.shape[1] - ids["input_ids"].shape[1]
    res = {"tokens_par_s": round(n / dt, 1),
           "vram_max_go": round(torch.cuda.max_memory_allocated() / 1e9, 2)}
    print(res); print(tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True)[:400])
    return res

In [ ]:
# 5. Qwen3-4B en FP16
free()
MID = "Qwen/Qwen3-4B"
tok = AutoTokenizer.from_pretrained(MID)
model = AutoModelForCausalLM.from_pretrained(MID, torch_dtype=torch.float16, device_map="cuda")
report["qwen3_4b_fp16"] = bench(model, tok)
del model; free(); save_report()

In [ ]:
# 6. Qwen3-8B en 4 bits (NF4, calcul en FP16)
MID = "Qwen/Qwen3-8B"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(MID)
model = AutoModelForCausalLM.from_pretrained(MID, quantization_config=bnb, device_map="cuda")
report["qwen3_8b_4bit"] = bench(model, tok)
del model; free(); save_report()

## 7. QLoRA : 50 pas sur un mini-jeu synthétique (texte de tableau → JSON)

Ce n'est **pas** l'entraînement du projet (S3-S4, bonus) : on vérifie seulement que la mémoire tient et que les checkpoints fonctionnent.
Rappel chapitre 6 : entraîner 1B paramètres en FP32 demande environ 24 Go, le T4 en a 16 → modèle gelé en 4 bits + adaptateurs LoRA.

In [ ]:
# 7a. Définitions (données, modèle, entraîneur) — à relancer après une coupure
import random
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

MID = "Qwen/Qwen3-4B"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(MID)
tok.pad_token = tok.pad_token or tok.eos_token

def make_example(rng):
    n = rng.randint(2, 5)
    prime = rng.randint(50, 500)
    sorties = [rng.randint(5, 150) for _ in range(n)]
    taux = rng.choice([0, 0.01, 0.025, 0.05])
    src = f"Primes : {prime} | Sinistres et frais : {', '.join(map(str, sorties))} | Taux : {taux:.1%}"
    tgt = json.dumps({"premiums": [prime] + [0] * (n - 1), "outflows": sorties,
                      "discount_rates": taux}, ensure_ascii=False)
    msgs = [{"role": "user", "content": "Extrais en JSON : " + src},
            {"role": "assistant", "content": tgt}]
    return {"text": tok.apply_chat_template(msgs, tokenize=False, enable_thinking=False)}

rng = random.Random(0)
ds = Dataset.from_list([make_example(rng) for _ in range(400)])
ds = ds.map(lambda b: tok(b["text"], truncation=True, max_length=256), batched=True,
            remove_columns=["text"])

def build_model():
    m = AutoModelForCausalLM.from_pretrained(MID, quantization_config=bnb, device_map="cuda")
    m = prepare_model_for_kbit_training(m, use_gradient_checkpointing=True)
    cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                     target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
    m = get_peft_model(m, cfg)
    m.print_trainable_parameters()
    return m

def build_trainer(model, max_steps):
    args = TrainingArguments(
        output_dir=CKPT, max_steps=max_steps, per_device_train_batch_size=4,
        gradient_accumulation_steps=2, learning_rate=2e-4, fp16=True, bf16=False,
        logging_steps=5, save_steps=25, save_total_limit=2, report_to="none",
        gradient_checkpointing=True, optim="paged_adamw_8bit")
    return Trainer(model=model, args=args, train_dataset=ds,
                   data_collator=DataCollatorForLanguageModeling(tok, mlm=False))

In [ ]:
# 7b. Entraînement de 50 pas
free()
model = build_model()
t0 = time.perf_counter()
trainer = build_trainer(model, max_steps=50)
out = trainer.train()
report["qlora_50_pas"] = {"duree_s": round(time.perf_counter() - t0, 1),
                          "loss_finale": round(out.training_loss, 4),
                          "vram_max_go": round(torch.cuda.max_memory_allocated() / 1e9, 2)}
print(report["qlora_50_pas"]); save_report()

## 8. Simuler une coupure puis reprendre

1. *Exécution → Redémarrer la session* (la mémoire est vidée, comme lors d'une vraie coupure).
2. Relancer les cellules **1, 2, 3, 4 et 7a** (pas 7b).
3. Lancer la cellule 9 : l'entraînement doit repartir du checkpoint du pas 50 et s'arrêter au pas 60.

In [ ]:
# 9. Reprise sur checkpoint
import glob
ckpts = sorted(glob.glob(f'{CKPT}/checkpoint-*'), key=lambda p: int(p.rsplit('-', 1)[1]))
print("checkpoints trouvés :", ckpts)
free()
model = build_model()
trainer = build_trainer(model, max_steps=60)
trainer.train(resume_from_checkpoint=True)
report["reprise"] = {"depuis": ckpts[-1].rsplit('/', 1)[1] if ckpts else None,
                     "pas_final": trainer.state.global_step,
                     "ok": trainer.state.global_step == 60 and bool(ckpts)}
print(report["reprise"]); del model; free(); save_report()

## 10. Exposer l'API FastAPI par un tunnel (sans Docker)

Utile si un modèle local tourne sur Colab. **Pour la démo, on reste sur un LLM par API** (risque de coupure, voir le planning §14).
Remplacez l'URL du dépôt avant d'exécuter.

In [ ]:
# 10. API + tunnel cloudflared (aucun compte requis pour un tunnel rapide)
REPO = "https://github.com/<organisation>/ifrs17-copilot.git"   # ← à remplacer
API_TOKEN = "change-moi"                                         # ← même valeur que dans n8n
!git clone -q $REPO /content/ifrs17-copilot 2>/dev/null || (cd /content/ifrs17-copilot && git pull -q)
%cd /content/ifrs17-copilot
!pip install -q -r requirements.txt
import subprocess, re, requests
env = dict(os.environ, API_TOKEN=API_TOKEN, LLM_PROVIDER="fake")
api = subprocess.Popen(["uvicorn", "api.main:app", "--port", "8000"], env=env,
                       stdout=open("/content/api.log", "w"), stderr=subprocess.STDOUT)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8000"],
                          stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(30):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m: url = m.group(0); break
print("URL publique :", url)
t0 = time.perf_counter()
r = requests.get(f"{url}/health", timeout=15)
report["tunnel"] = {"url": url, "statut": r.status_code, "aller_retour_s": round(time.perf_counter() - t0, 2)}
print(report["tunnel"], r.json()); save_report()

## 11. Bilan à recopier dans `docs/decisions.md`

Affichez le rapport et cochez chaque ligne du tableau go / no-go du début. Toute ligne rouge déclenche le plan B prévu (modèle plus petit, API seule).

In [ ]:
print(json.dumps(report, indent=2, ensure_ascii=False))